# **`mqqr` &mdash; full tutorial on real US macroeconomic data**

**Multivariate Quantile-on-Quantile Regression & Causality**

*Author: Dr. Merwan Roudane &middot; `merwanroudane920@gmail.com` &middot; [github.com/merwanroudane/qqrpy](https://github.com/merwanroudane/qqrpy)*

---

This notebook walks through the full `mqqr` library on real **quarterly US macroeconomic data** (1959 Q1 - 2009 Q3) from `statsmodels.datasets.macrodata`. We run five complete econometric workflows:

| # | Workflow                                                                 | Function                       |
|---|--------------------------------------------------------------------------|--------------------------------|
| 1 | Bivariate QQR &mdash; Phillips curve at quantiles                        | `qq_regression`                |
| 2 | Additive m-QQR (Type 1, Alola et al. 2023) &mdash; three drivers of INFL | `additive_mqq_regression`      |
| 3 | Moderated m-QQR (Type 2, Sinha et al. 2023) &mdash; main + moderators    | `mqq_regression`               |
| 4 | Bivariate QQ Granger causality                                           | `qq_causality`                 |
| 5 | Conditional (m-QQ) Granger causality                                     | `mqq_causality`                |

All figures use the **MATLAB Jet** colour-map at 300 dpi; all tables follow Elsevier / Springer journal format with `*`, `**`, `***` significance markers.

## 1. Setup &amp; data preparation

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.datasets.macrodata as macrodata

import mqqr
print(f'mqqr version: {mqqr.__version__}')

FIGDIR = 'docs/images'
TABDIR = 'docs/tables'
os.makedirs(FIGDIR, exist_ok=True)
os.makedirs(TABDIR, exist_ok=True)

# 9-point quantile grid (0.10, 0.20, ..., 0.90) -- journal-grade visuals
# but ~5x faster than the 19-point default for bootstrap-heavy estimators
QGRID = np.arange(0.10, 0.91, 0.10)
print(f'Quantile grid: {np.round(QGRID, 2)}')

In [ ]:
raw = macrodata.load_pandas().data.copy()
raw['date'] = pd.PeriodIndex(
    year=raw['year'].astype(int),
    quarter=raw['quarter'].astype(int),
    freq='Q',
).to_timestamp()
raw = raw.set_index('date')

df = pd.DataFrame(index=raw.index)
df['INFL']   = raw['infl']                                  # CPI inflation (%)
df['GDP_GR'] = 100.0 * np.log(raw['realgdp']).diff()        # real GDP growth (%)
df['M1_GR']  = 100.0 * np.log(raw['m1']).diff()             # M1 growth (%)
df['UNEMP']  = raw['unemp']                                 # unemployment (%)
df['TBILL']  = raw['tbilrate']                              # 3-month T-bill (%)
df = df.dropna()

print(f'Sample: {df.index.min().date()} -- {df.index.max().date()}  (n = {len(df)})')
df.head()

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(11, 9), sharex=True, facecolor='white')
colors = ['#d62728', '#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e']
labels = ['Inflation (%)', 'Real GDP growth (%)', 'M1 growth (%)',
          'Unemployment (%)', '3-month T-bill (%)']
for ax, col, c, lab in zip(axes, df.columns, colors, labels):
    ax.plot(df.index, df[col], color=c, lw=1.2)
    ax.set_ylabel(lab, fontsize=9)
    ax.grid(True, alpha=0.3, linewidth=0.4)
axes[-1].set_xlabel('Quarter')
fig.suptitle('US quarterly macroeconomic series  (1959 Q2 - 2009 Q3)',
             fontsize=13, fontweight='bold', y=1.00)
fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig00_series_overview.png', dpi=300, bbox_inches='tight')
plt.show()

## 2.  Descriptive statistics (Jarque-Bera + ARCH-LM)

In [ ]:
desc = mqqr.descriptive_table({c: df[c].values for c in df.columns})
desc

In [ ]:
with open(f'{TABDIR}/tab01_descriptive.md', 'w', encoding='utf-8') as f:
    f.write('# Descriptive statistics\n\n')
    f.write(desc.to_markdown(floatfmt='.4f'))
with open(f'{TABDIR}/tab01_descriptive.tex', 'w', encoding='utf-8') as f:
    f.write(desc.to_latex(float_format='%.4f',
                          caption='Descriptive statistics for US macro series',
                          label='tab:desc'))
desc.to_csv(f'{TABDIR}/tab01_descriptive.csv')
print('Saved tab01_descriptive.{md,tex,csv}')

## 3.  Workflow 1 &mdash; Bivariate QQR (Sim &amp; Zhou, 2015)

### Phillips curve at quantiles: does **unemployment** affect **inflation** differently across the inflation distribution?

$$
\mathrm{INFL}_t \;=\; \beta_0(\theta,\tau) \;+\; \beta_1(\theta,\tau)\bigl(\mathrm{UNEMP}_t-\mathrm{UNEMP}^{\tau}\bigr) \;+\; \alpha^\theta\,\mathrm{INFL}_{t-1} \;+\; v_t^\theta
$$

In [ ]:
qq = mqqr.qq_regression(
    y=df['INFL'].values, x=df['UNEMP'].values,
    y_quantiles=QGRID, x_quantiles=QGRID,
    bandwidth=0.05, n_boot=60,
    x_name='UNEMP', y_name='INFL',
    verbose=False,
)
qq.summary()

In [ ]:
fig, ax = mqqr.plot_qq_3d(
    qq, cmap='jet', elev=28, azim=-130,
    title=r'Phillips curve at quantiles -- UNEMP $\to$ INFL',
    save_path=f'{FIGDIR}/fig01_qqr_3d.png',
)
plt.show()

In [ ]:
fig, ax = mqqr.plot_qq_heatmap(
    qq, cmap='jet', annotate='stars',
    title='QQR heat-map  (UNEMP $\\to$ INFL)',
    save_path=f'{FIGDIR}/fig02_qqr_heatmap.png',
)
plt.show()

In [ ]:
fig, ax = mqqr.plot_qq_contour(
    qq, cmap='jet',
    title='QQR contour  (UNEMP $\\to$ INFL)',
    save_path=f'{FIGDIR}/fig03_qqr_contour.png',
)
plt.show()

In [ ]:
tbl_qq = mqqr.results_table(qq, value='beta1', digits=4)
tbl_qq.to_csv(f'{TABDIR}/tab02_qqr_beta1.csv')
with open(f'{TABDIR}/tab02_qqr_beta1.md', 'w', encoding='utf-8') as f:
    f.write('# QQR results -- $\\hat{\\beta}_1(\\theta,\\tau)$\n\n')
    f.write(mqqr.to_markdown(tbl_qq, caption='UNEMP -> INFL'))
with open(f'{TABDIR}/tab02_qqr_beta1.tex', 'w', encoding='utf-8') as f:
    f.write(mqqr.to_latex(tbl_qq,
                          caption=r'QQR slope $\hat{\beta}_1$ -- UNEMP $\to$ INFL',
                          label='tab:qqr',
                          notes=r'Stars: $^*p<0.10,\ ^{**}p<0.05,\ ^{***}p<0.01$.'))
tbl_qq

## 4.  Workflow 2 &mdash; Additive m-QQR  (Type 1, Alola et al. 2023)

### Three drivers of inflation, each with its **own** quantile axis $\Phi_i$, no interactions

$$
\mathrm{INFL}_t = \beta_0(\theta,\Phi_1,\Phi_2,\Phi_3) + \sum_{i\in\{\text{GDP\_GR},\,\text{M1\_GR},\,\text{UNEMP}\}}\beta_i(\theta,\Phi_i)(x_{i,t}-x_i^{\Phi_i}) + \alpha^\theta\,\mathrm{INFL}_{t-1} + \epsilon_t^\theta
$$

In [ ]:
amq = mqqr.additive_mqq_regression(
    y=df['INFL'].values,
    X={
        'GDP_GR': df['GDP_GR'].values,
        'M1_GR':  df['M1_GR'].values,
        'UNEMP':  df['UNEMP'].values,
    },
    y_quantiles=QGRID, x_quantiles=QGRID,
    bandwidth=0.05, n_boot=40,
    y_name='INFL',
    verbose=False,
)
amq.summary()

In [ ]:
fig = mqqr.plot_additive_mqq_panel(
    amq, cmap='jet',
    save_path=f'{FIGDIR}/fig04_additive_mqqr_panel.png',
)
plt.show()

In [ ]:
for name in amq.x_names:
    fig, ax = mqqr.plot_additive_mqq_3d(
        amq, name, cmap='jet',
        save_path=f'{FIGDIR}/fig05_amqqr_{name}_3d.png',
    )
    plt.show()
    plt.close(fig)

In [ ]:
from mqqr import significance_stars

for name in amq.x_names:
    V = amq.to_matrix(name, 'beta')
    P = amq.to_matrix(name, 'p_value')
    yq = np.asarray(amq.y_quantiles, dtype=float)
    xq = np.asarray(amq.x_quantiles, dtype=float)
    cells = np.full(V.shape, '', dtype=object)
    for i in range(V.shape[0]):
        for j in range(V.shape[1]):
            v = V[i, j]
            if np.isfinite(v):
                cells[i, j] = f'{v:+.4f}' + significance_stars(P[i, j])
            else:
                cells[i, j] = '--'
    out = pd.DataFrame(
        cells,
        index=[f'{q:.2f}' for q in yq],
        columns=[f'{q:.2f}' for q in xq],
    )
    out.index.name = 'Y_q \\ X_q'
    out.to_csv(f'{TABDIR}/tab03_amqqr_{name}.csv')
    with open(f'{TABDIR}/tab03_amqqr_{name}.md', 'w', encoding='utf-8') as f:
        f.write(f'# Additive m-QQR -- {name} $\\to$ INFL\n\n')
        f.write(mqqr.to_markdown(out))
    with open(f'{TABDIR}/tab03_amqqr_{name}.tex', 'w', encoding='utf-8') as f:
        f.write(mqqr.to_latex(out,
                              caption=f'Additive m-QQR: {name} $\\to$ INFL',
                              label=f'tab:amqqr_{name}',
                              notes=r'Stars: $^*p<0.10,\ ^{**}p<0.05,\ ^{***}p<0.01$.'))
    print(f'Saved tab03_amqqr_{name}')
out

## 5.  Workflow 3 &mdash; Moderated m-QQR  (Type 2, Sinha et al. 2023)

### **GDP growth** $\to$ **inflation**, moderated by **unemployment** and **M1 growth**, with $x\cdot Z$ interactions

$$
\mathrm{INFL}_t = \beta_0(\theta,\tau) + \beta_1(\theta,\tau)(\mathrm{GDP\_GR}_t-\mathrm{GDP\_GR}^\tau) + \sum_{j\in\{\text{UNEMP},\text{M1\_GR}\}}\!\!\bigl[\gamma_j(\theta,\tau)\,\mathrm{GDP\_GR}_t\cdot Z_{j,t} + \alpha_j(\theta)\,Z_{j,t}\bigr] + \epsilon_t^\theta
$$

In [ ]:
mq = mqqr.mqq_regression(
    y=df['INFL'].values,
    x=df['GDP_GR'].values,
    moderators={
        'UNEMP': df['UNEMP'].values,
        'M1_GR': df['M1_GR'].values,
    },
    y_quantiles=QGRID, x_quantiles=QGRID,
    bandwidth=0.05, n_boot=40,
    x_name='GDP_GR', y_name='INFL',
    verbose=False,
)
mq.summary()

In [ ]:
fig = mqqr.plot_mqq_panel(
    mq, cmap='jet',
    save_path=f'{FIGDIR}/fig06_mqqr_panel.png',
)
plt.show()

In [ ]:
fig, ax = mqqr.plot_mqq_3d(
    mq, cmap='jet',
    title=r'Moderated m-QQR -- GDP$\_$GR $\to$ INFL',
    save_path=f'{FIGDIR}/fig07_mqqr_principal_3d.png',
)
plt.show()

In [ ]:
tbl_mq = mqqr.results_table(mq, value='beta1', digits=4)
tbl_mq.to_csv(f'{TABDIR}/tab04_mqqr_beta1.csv')
with open(f'{TABDIR}/tab04_mqqr_beta1.md', 'w', encoding='utf-8') as f:
    f.write('# Moderated m-QQR -- principal slope $\\hat{\\beta}_1$\n\n')
    f.write(mqqr.to_markdown(tbl_mq, caption='GDP_GR -> INFL'))
with open(f'{TABDIR}/tab04_mqqr_beta1.tex', 'w', encoding='utf-8') as f:
    f.write(mqqr.to_latex(tbl_mq,
                          caption=r'Moderated m-QQR principal slope $\hat{\beta}_1$ -- GDP$\_$GR $\to$ INFL',
                          label='tab:mqqr_beta1',
                          notes=r'Stars: $^*p<0.10,\ ^{**}p<0.05,\ ^{***}p<0.01$.'))
tbl_mq

In [ ]:
for name in mq.moderator_names:
    G = mq.interaction_matrix(name, 'gamma')
    sub = mq.interactions[mq.interactions['moderator'] == name]
    P = sub.pivot(index='y_quantile', columns='x_quantile', values='p_value') \
           .sort_index(axis=0).sort_index(axis=1).values
    yq = np.asarray(mq.y_quantiles, dtype=float)
    xq = np.asarray(mq.x_quantiles, dtype=float)
    cells = np.full(G.shape, '', dtype=object)
    for i in range(G.shape[0]):
        for j in range(G.shape[1]):
            v = G[i, j]
            if np.isfinite(v):
                cells[i, j] = f'{v:+.4f}' + significance_stars(P[i, j])
            else:
                cells[i, j] = '--'
    out = pd.DataFrame(
        cells,
        index=[f'{q:.2f}' for q in yq],
        columns=[f'{q:.2f}' for q in xq],
    )
    out.index.name = 'Y_q \\ X_q'
    out.to_csv(f'{TABDIR}/tab05_mqqr_gamma_{name}.csv')
    with open(f'{TABDIR}/tab05_mqqr_gamma_{name}.md', 'w', encoding='utf-8') as f:
        f.write(f'# Moderation $\\gamma$ -- {name}\n\n')
        f.write(mqqr.to_markdown(out))
    with open(f'{TABDIR}/tab05_mqqr_gamma_{name}.tex', 'w', encoding='utf-8') as f:
        f.write(mqqr.to_latex(out,
                              caption=f'Moderation $\\gamma$ -- {name}',
                              label=f'tab:mqqr_gamma_{name}'))
    print(f'Saved tab05_mqqr_gamma_{name}')

## 6.  Workflow 4 &mdash; Bivariate QQ Granger causality

In [ ]:
cq = mqqr.qq_causality(
    x=df['GDP_GR'].values, y=df['INFL'].values,
    y_quantiles=QGRID, x_quantiles=QGRID,
    bandwidth=0.05, n_boot=80,
    cause_name='GDP_GR', effect_name='INFL',
    verbose=False,
)
cq.summary()

In [ ]:
fig, ax = mqqr.plot_qq_causality_heatmap(
    cq, cmap='red_yellow_black', show_stars=True,
    title=r'QQ Granger causality -- GDP$\_$GR $\to$ INFL',
    save_path=f'{FIGDIR}/fig08_qq_causality.png',
)
plt.show()

## 7.  Workflow 5 &mdash; Conditional (m-QQ) Granger causality

In [ ]:
mcq = mqqr.mqq_causality(
    x=df['GDP_GR'].values, y=df['INFL'].values,
    moderators={
        'UNEMP': df['UNEMP'].values,
        'M1_GR': df['M1_GR'].values,
    },
    y_quantiles=QGRID, x_quantiles=QGRID,
    bandwidth=0.05, n_boot=50,
    cause_name='GDP_GR', effect_name='INFL',
    verbose=False,
)
mcq.summary()

In [ ]:
fig, ax = mqqr.plot_qq_causality_heatmap(
    mcq, cmap='red_yellow_black', show_stars=True,
    title=r'Conditional m-QQ causality -- GDP$\_$GR $\to$ INFL  (cond. UNEMP, M1$\_$GR)',
    save_path=f'{FIGDIR}/fig09_mqq_causality.png',
)
plt.show()

## 8.  Colour-map gallery

In [ ]:
fig = mqqr.show_colormaps()
fig.savefig(f'{FIGDIR}/fig10_colormaps.png', dpi=300, bbox_inches='tight')
plt.show()

## 9.  Done

All figures and tables were saved into `docs/images/` and `docs/tables/`, ready for GitHub Pages.

---

**License:** MIT &copy; 2026 Dr. Merwan Roudane  
**GitHub:** <https://github.com/merwanroudane/qqrpy>  
**PyPI:** <https://pypi.org/project/mqqr/>